# 04 â€” Results

Loads metrics for all four models (RF, XGBoost, PTT-LR, CNN), renders:

1. **3-class comparison table** â€” accuracy, macro-F1, hypertensive AUROC, hypertensive recall, false-negative rate.
2. **Confusion matrix** for the best model on the held-out test set.
3. **Feature attribution** â€” top permutation-importance features for RF and XGBoost.
4. **CNN saliency plot** â€” `(2, 1000)` mean |gradient| over the test loader, overlaid on the mean PPG/ECG waveform so peaks line up with actual signal landmarks.
5. **Auto-filled writeup paragraph** â€” substitutes the loaded metrics into the target paragraph from the project brief.

The partner-SVM block is gated on `svm_metrics.json`; uncomment once available.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from bme_ml.paths import setup_paths
from bme_ml.labels import MULTICLASS_NAMES
from bme_ml.evaluation import MultiClassMetrics, compare_multiclass
from bme_ml.saliency import plot_saliency

paths = setup_paths()


def load_metrics(name: str) -> MultiClassMetrics | None:
    p = paths.processed / f'{name}_metrics.json'
    if not p.exists():
        return None
    return MultiClassMetrics(**json.loads(p.read_text()))


metrics = {
    'Random Forest': load_metrics('rf'),
    'XGBoost': load_metrics('xgb'),
    'PTT-only LR': load_metrics('ptt_lr'),
    '1D CNN': load_metrics('cnn'),
}
for k, v in metrics.items():
    print(f'{k:15s} -> {"loaded" if v is not None else "MISSING"}')

# Optional partner SVM â€” uncomment once the partner drops svm_metrics.json.
# svm_metrics = load_metrics('svm')
# if svm_metrics is not None:
#     metrics['SVM (partner)'] = svm_metrics

In [ ]:
# Comparison table.
rows = [(k, v) for k, v in metrics.items() if v is not None]
table = compare_multiclass(rows).sort_values('hypertensive_auroc', ascending=False).reset_index(drop=True)
table

In [ ]:
# Confusion matrix for the best model (by hypertensive AUROC).
best_name = table.iloc[0]['model']
best_metrics = metrics[best_name]
cm = np.array(best_metrics.confusion)

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(cm, display_labels=MULTICLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'{best_name} â€” 3-class confusion (held-out test)')
plt.tight_layout(); plt.show()

# Per-class recall printout for the writeup paragraph.
per_class_recall = cm.diagonal() / cm.sum(axis=1).clip(min=1)
for name, r in zip(MULTICLASS_NAMES, per_class_recall):
    print(f'{name:14s}  recall={r:.3f}')

In [ ]:
# Feature attribution for the tabular models â€” top 8 permutation features.
imp_paths = {
    'Random Forest': paths.processed / 'rf_importance.csv',
    'XGBoost':       paths.processed / 'xgb_importance.csv',
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
top_features = {}
for ax, (model_name, p) in zip(axes, imp_paths.items()):
    if not p.exists():
        ax.set_visible(False); continue
    df = pd.read_csv(p).head(8)
    top_features[model_name] = df['feature'].tolist()
    ax.barh(df['feature'][::-1], df['importance_mean'][::-1], xerr=df['importance_std'][::-1], color='C0', alpha=0.8)
    ax.set_title(f'{model_name} â€” top 8 (permutation)')
    ax.set_xlabel('Î” score when shuffled')
plt.tight_layout(); plt.show()
top_features

In [ ]:
# CNN saliency â€” which time regions of the waveform did the CNN focus on.
sal_path = paths.processed / 'cnn_saliency.npy'
wave_path = paths.processed / 'cnn_mean_waveform.npy'
top_temporal_regions = []
if sal_path.exists() and wave_path.exists():
    sal = np.load(sal_path)
    wave = np.load(wave_path)
    plot_saliency(sal, wave)
    plt.suptitle('CNN saliency overlaid on mean test-set waveform', y=1.02)
    plt.tight_layout(); plt.show()

    # Identify the highest-saliency time regions per channel for the writeup.
    SAMPLE_RATE_HZ = 125.0
    for c, name in enumerate(['PPG', 'ECG']):
        peak_idx = int(np.argmax(sal[c]))
        peak_t_s = peak_idx / SAMPLE_RATE_HZ
        # Width at half-max around the peak.
        half = sal[c, peak_idx] * 0.5
        left = peak_idx
        while left > 0 and sal[c, left - 1] >= half:
            left -= 1
        right = peak_idx
        while right < len(sal[c]) - 1 and sal[c, right + 1] >= half:
            right += 1
        region = (left / SAMPLE_RATE_HZ, right / SAMPLE_RATE_HZ)
        top_temporal_regions.append((name, peak_t_s, region))
        print(f'{name}: peak |grad| at t={peak_t_s:.2f}s, FWHM region {region[0]:.2f}-{region[1]:.2f}s')
else:
    print('saliency artifacts missing â€” re-run notebook 05')

In [ ]:
# Auto-fill the writeup paragraph from the loaded metrics.
ptt = metrics['PTT-only LR']
tabular_features = ', '.join(top_features.get('XGBoost') or top_features.get('Random Forest') or [])[:120]
saliency_desc = '; '.join(
    f'{name} (peak ~{peak:.2f}s, {r[0]:.2f}-{r[1]:.2f}s window)'
    for name, peak, r in top_temporal_regions
) if top_temporal_regions else 'saliency unavailable'

paragraph = (
    f"The best performing model was a **{best_name}** which achieved an overall accuracy of "
    f"{best_metrics.accuracy*100:.1f}% and an F1 score of {best_metrics.f1_macro:.3f} on the "
    f"held-out test set. The AUROC for detecting hypertensive cases was "
    f"{best_metrics.hypertensive_auroc:.3f}, indicating "
    f"{'strong' if best_metrics.hypertensive_auroc >= 0.80 else 'moderate' if best_metrics.hypertensive_auroc >= 0.70 else 'limited'} "
    f"discriminative ability. Compared to a baseline feature-based model â€” PTT-only logistic "
    f"regression â€” which achieved an accuracy of {ptt.accuracy*100:.1f}% and an AUROC of "
    f"{ptt.hypertensive_auroc:.3f}, our proposed model demonstrated "
    f"{'consistent improvement' if best_metrics.hypertensive_auroc > ptt.hypertensive_auroc else 'no improvement'} "
    f"across all metrics. Confusion matrix analysis showed that the model correctly identified "
    f"{best_metrics.hypertensive_recall*100:.1f}% of hypertensive cases with a false negative "
    f"rate of {best_metrics.hypertensive_false_negative_rate*100:.1f}%. Feature attribution analysis "
    f"indicates that {tabular_features} were among the most influential predictors in the feature-"
    f"based models, while the deep learning model focused on {saliency_desc} of the waveform."
)
from IPython.display import Markdown
Markdown(paragraph)